In [1]:
import os

ROOT = "/kaggle/input/datasets/artemmmtry/diode-a-dense-indoor-and-outdoor-depth-dataset"

print("Cấp 1 (train/val):", os.listdir(ROOT))

train_path = os.path.join(ROOT, "train")
domains = os.listdir(train_path)
print("Cấp 2 (domain indoor/outdoor):", domains)

# Tự động nhận diện domain indoor và outdoor dù tên viết khác nhau
indoor_name = next((d for d in domains if "indoor" in d.lower()), None)
outdoor_name = next((d for d in domains if "outdoor" in d.lower()), None)
print(f"\nĐã nhận diện: indoor='{indoor_name}', outdoor='{outdoor_name}'")

# Xem thử 1 scene có gì bên trong
sample_scene_path = os.path.join(train_path, indoor_name)
scenes = sorted(os.listdir(sample_scene_path))
print(f"\nIndoor có {len(scenes)} scene, ví dụ: {scenes[:3]}")
scans = sorted(os.listdir(os.path.join(sample_scene_path, scenes[0])))
print(f"Scene '{scenes[0]}' có {len(scans)} scan: {scans}")
files = os.listdir(os.path.join(sample_scene_path, scenes[0], scans[0]))
print(f"Trong 1 scan có các file: {files[:6]}")

Cấp 1 (train/val): ['val', 'train']
Cấp 2 (domain indoor/outdoor): ['outdoor', 'indoors']

Đã nhận diện: indoor='indoors', outdoor='outdoor'

Indoor có 7 scene, ví dụ: ['scene_00000', 'scene_00001', 'scene_00002']
Scene 'scene_00000' có 8 scan: ['scan_00000', 'scan_00001', 'scan_00002', 'scan_00003', 'scan_00004', 'scan_00005', 'scan_00006', 'scan_00007']
Trong 1 scan có các file: ['00000_00000_indoors_050_050.png', '00000_00000_indoors_120_050_depth_mask.npy', '00000_00000_indoors_110_040_depth.npy', '00000_00000_indoors_090_020_depth_mask.npy', '00000_00000_indoors_090_040.png', '00000_00000_indoors_060_020.png']


In [2]:
def collect_pairs(domain_path, domain_name):
    pairs = []
    scenes = sorted(os.listdir(domain_path))
    for scene in scenes:
        scene_path = os.path.join(domain_path, scene)
        if not os.path.isdir(scene_path):
            continue
        scans = sorted(os.listdir(scene_path))
        for scan in scans:
            scan_path = os.path.join(scene_path, scan)
            if not os.path.isdir(scan_path):
                continue
            all_files = os.listdir(scan_path)
            rgb_files = [f for f in all_files if f.endswith(".png")]
            for rgb_file in rgb_files:
                base = rgb_file.replace(".png", "")
                depth_file = f"{base}_depth.npy"
                mask_file = f"{base}_depth_mask.npy"
                if depth_file in all_files and mask_file in all_files:
                    pairs.append({
                        "rgb": os.path.join(scan_path, rgb_file),
                        "depth": os.path.join(scan_path, depth_file),
                        "mask": os.path.join(scan_path, mask_file),
                        "domain": domain_name,
                        "scene": scene,
                        "scan": scan
                    })
    return pairs

indoor_pairs = collect_pairs(os.path.join(train_path, indoor_name), "indoor")
outdoor_pairs = collect_pairs(os.path.join(train_path, outdoor_name), "outdoor")

print(f"Indoor: {len(indoor_pairs)} pairs, {len(set(p['scene'] for p in indoor_pairs))} scenes")
print(f"Outdoor: {len(outdoor_pairs)} pairs, {len(set(p['scene'] for p in outdoor_pairs))} scenes")

Indoor: 8574 pairs, 7 scenes
Outdoor: 16884 pairs, 12 scenes


In [3]:
import random
random.seed(42)

def sample_by_scene_capped(pairs, target_count, max_per_scene):
    scenes = sorted(set(p["scene"] for p in pairs))
    random.shuffle(scenes)
    selected, used_scenes = [], []
    for scene in scenes:
        if len(selected) >= target_count:
            break
        scene_pairs = [p for p in pairs if p["scene"] == scene]
        random.shuffle(scene_pairs)
        take = scene_pairs[:max_per_scene]  # chỉ lấy tối đa max_per_scene ảnh/scene
        selected.extend(take)
        used_scenes.append(scene)
    return selected, used_scenes

# Indoor: 7 scene tổng, muốn dùng gần hết để có đủ scene chia train/val/test
indoor_subset, indoor_scenes_used = sample_by_scene_capped(
    indoor_pairs, target_count=4000, max_per_scene=700
)

# Outdoor: 12 scene tổng, giới hạn mạnh hơn để bắt buộc dùng nhiều scene
outdoor_subset, outdoor_scenes_used = sample_by_scene_capped(
    outdoor_pairs, target_count=3000, max_per_scene=400
)

all_subset = indoor_subset + outdoor_subset
print(f"Tổng subset: {len(all_subset)} pairs")
print(f"Scene dùng: {len(indoor_scenes_used)}/7 indoor + {len(outdoor_scenes_used)}/12 outdoor")

Tổng subset: 7195 pairs
Scene dùng: 7/7 indoor + 8/12 outdoor


In [4]:
random.seed(42)

def split_scenes(scenes_list, train_ratio=0.7, val_ratio=0.15):
    scenes = scenes_list.copy()
    random.shuffle(scenes)
    n = len(scenes)
    n_train = max(1, round(n * train_ratio))
    n_val = max(1, round(n * val_ratio))
    if n_train + n_val >= n:
        n_train = max(1, n - 2)
        n_val = 1
    train_scenes = scenes[:n_train]
    val_scenes = scenes[n_train:n_train + n_val]
    test_scenes = scenes[n_train + n_val:]
    return train_scenes, val_scenes, test_scenes

indoor_train, indoor_val, indoor_test = split_scenes(indoor_scenes_used)
outdoor_train, outdoor_val, outdoor_test = split_scenes(outdoor_scenes_used)

print("Indoor  -> train:", indoor_train, "| val:", indoor_val, "| test:", indoor_test)
print("Outdoor -> train:", outdoor_train, "| val:", outdoor_val, "| test:", outdoor_test)

def assign_split(pairs, train_scenes, val_scenes, test_scenes):
    for p in pairs:
        if p["scene"] in train_scenes:
            p["split"] = "train"
        elif p["scene"] in val_scenes:
            p["split"] = "val"
        elif p["scene"] in test_scenes:
            p["split"] = "test"
    return pairs

indoor_subset = assign_split(indoor_subset, indoor_train, indoor_val, indoor_test)
outdoor_subset = assign_split(outdoor_subset, outdoor_train, outdoor_val, outdoor_test)

all_subset = indoor_subset + outdoor_subset

# Thống kê số lượng pairs theo từng split
from collections import Counter
split_counts = Counter(p["split"] for p in all_subset)
print("\nSố pairs theo split:", dict(split_counts))

Indoor  -> train: ['scene_00003', 'scene_00002', 'scene_00006', 'scene_00004', 'scene_00005'] | val: ['scene_00001'] | test: ['scene_00000']
Outdoor -> train: ['scene_00013', 'scene_00007', 'scene_00015', 'scene_00008', 'scene_00014', 'scene_00010'] | val: ['scene_00017'] | test: ['scene_00018']

Số pairs theo split: {'val': 1100, 'train': 4995, 'test': 1100}


In [ ]:
from PIL import Image
import numpy as np
import cv2

TARGET_SIZE = (384, 288)  # giảm từ 1024x768 xuống ~1/7 dung lượng, vẫn đủ chi tiết để train

OUT_DIR = "/kaggle/working/diode_subset"
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(OUT_DIR, split), exist_ok=True)

def resize_and_save(p, split):
    base = os.path.basename(p["rgb"]).replace(".png", "")
    out_folder = os.path.join(OUT_DIR, split)

    # RGB la FEATURE: ha mau muot khong lam hong gi, va JPEG giu subset nho.
    img = Image.open(p["rgb"]).convert("RGB").resize(TARGET_SIZE, Image.BILINEAR)
    img.save(os.path.join(out_folder, f"{base}.jpg"), quality=90)

    # Depth la LABEL, bat buoc NEAREST.
    # DIODE luu pixel invalid bang 0. Bilinear tron nhung so 0 do vao pixel valid lan can
    # va keo gia tri xuong, sinh ra khoang cach khong co be mat nao nam o do. Mask dung
    # NEAREST nen van sach, tuc no KHONG danh dau duoc cho bi nhiem: mot pixel co the
    # mask = 1 trong khi depth da bi trung binh voi vung invalid.
    depth = np.load(p["depth"]).squeeze()
    depth_resized = cv2.resize(depth, TARGET_SIZE, interpolation=cv2.INTER_NEAREST)
    np.save(os.path.join(out_folder, f"{base}_depth.npy"), depth_resized.astype(np.float16))

    # Mask: nearest, giu nhi phan 0/1 dung, khong bi noi suy mo bien
    mask = np.load(p["mask"]).squeeze()
    mask_resized = cv2.resize(mask.astype(np.uint8), TARGET_SIZE, interpolation=cv2.INTER_NEAREST)
    np.save(os.path.join(out_folder, f"{base}_depth_mask.npy"), mask_resized.astype(np.uint8))

for i, p in enumerate(all_subset):
    resize_and_save(p, p["split"])
    if i % 1000 == 0:
        print(f"Đã xử lý {i}/{len(all_subset)}")

print("Xong!")
os.system(f"du -sh {OUT_DIR}")


In [6]:
import pandas as pd

metadata = pd.DataFrame(all_subset)
metadata.to_csv("/kaggle/working/subset_metadata.csv", index=False)

scene_split_info = pd.DataFrame([
    {"domain": "indoor", "scene": s, "split": "train"} for s in indoor_train
] + [
    {"domain": "indoor", "scene": s, "split": "val"} for s in indoor_val
] + [
    {"domain": "indoor", "scene": s, "split": "test"} for s in indoor_test
] + [
    {"domain": "outdoor", "scene": s, "split": "train"} for s in outdoor_train
] + [
    {"domain": "outdoor", "scene": s, "split": "val"} for s in outdoor_val
] + [
    {"domain": "outdoor", "scene": s, "split": "test"} for s in outdoor_test
])
scene_split_info.to_csv("/kaggle/working/scenes_used.csv", index=False)

print("Đã lưu metadata. Preview scene_split:")
print(scene_split_info)
print(f"\nTổng metadata rows: {len(metadata)}")

Đã lưu metadata. Preview scene_split:
     domain        scene  split
0    indoor  scene_00003  train
1    indoor  scene_00002  train
2    indoor  scene_00006  train
3    indoor  scene_00004  train
4    indoor  scene_00005  train
5    indoor  scene_00001    val
6    indoor  scene_00000   test
7   outdoor  scene_00013  train
8   outdoor  scene_00007  train
9   outdoor  scene_00015  train
10  outdoor  scene_00008  train
11  outdoor  scene_00014  train
12  outdoor  scene_00010  train
13  outdoor  scene_00017    val
14  outdoor  scene_00018   test

Tổng metadata rows: 7195


In [7]:
import shutil

shutil.make_archive("/kaggle/working/diode_subset", "zip", "/kaggle/working/diode_subset")
os.system("du -sh /kaggle/working/diode_subset.zip")
os.system("du -sh /kaggle/working")  # kiểm tra tổng dung lượng working dir còn trống bao nhiêu

718M	/kaggle/working/diode_subset.zip
3.2G	/kaggle/working


0

In [ ]:
# Recompute the Section 2.4 table from the rebuilt subset.
# Percentiles come from a pooled histogram rather than from stacking every array, so this
# stays within Kaggle RAM. Paste the printed markdown straight into the proposal page.

import numpy as np, os
from collections import defaultdict

ROOT = "/kaggle/working/diode_subset"
SPLITS = ["train", "val", "test"]
BINS = np.arange(0, 400.05, 0.05)

def domain_of(sid):
    return "indoor" if "_indoors_" in sid else "outdoor"

def pct_from_hist(hist, q):
    cdf = np.cumsum(hist) / hist.sum()
    return BINS[min(np.searchsorted(cdf, q / 100.0) + 1, len(BINS) - 1)]

hists = defaultdict(lambda: np.zeros(len(BINS) - 1))
invalid = defaultdict(list)
counts = defaultdict(int)
mask_values = set()

for split in SPLITS:
    d = os.path.join(ROOT, split)
    ids = sorted(f[:-4] for f in os.listdir(d) if f.endswith(".jpg"))
    for sid in ids:
        dom = domain_of(sid)
        depth = np.load(os.path.join(d, sid + "_depth.npy")).astype(np.float32).squeeze()
        mask = np.load(os.path.join(d, sid + "_depth_mask.npy")).squeeze()
        mask_values.update(np.unique(mask).tolist())
        valid = (mask > 0) & np.isfinite(depth) & (depth > 0)
        hists[(split, dom)] += np.histogram(depth[valid], BINS)[0]
        invalid[(split, dom)].append(100.0 * (1.0 - valid.mean()))
        counts[(split, dom)] += 1
    print(split, "done")

print("\nmask distinct values over the whole subset:", sorted(mask_values))
print("(must be exactly [0, 1] - anything else means the mask was interpolated)\n")

print("| Split | Domain | Samples | p5 | Median | p95 | p99 | Mean Invalid % |")
print("|---|---|---|---|---|---|---|---|")
for split in SPLITS:
    for dom in ("indoor", "outdoor"):
        h = hists[(split, dom)]
        if h.sum() == 0:
            continue
        p5, p50, p95, p99 = [pct_from_hist(h, q) for q in (5, 50, 95, 99)]
        inv = float(np.mean(invalid[(split, dom)]))
        print("| %s | %s | %d | %.1fm | %.1fm | %.1fm | %.1fm | %.1f%% |"
              % (split, dom, counts[(split, dom)], p5, p50, p95, p99, inv))
